In [ ]:
from dlfs.layers import DenseLayer
from dlfs.activation import ReLU, Sigmoid, Softmax
from dlfs.loss import BCE_Loss, CCE_Loss
from dlfs.optimizers import Optimizer_SGD, Optimizer_Adam
from dlfs.model import SequentialModel

from sklearn.datasets import make_circles, make_moons, make_blobs

from viz_helpers import *

# Dense Layer visualisation for classification

- this notebook aims to visualize 1D, 2D, 3D linear transformations (by Dense Layers) and activation functions applied to classification data

# Different datasets and layers for visualisation

In [ ]:
def make_xor():

    X = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])
    y = np.array([0, 1, 1, 0])

    return X, y

circles_layers =  [DenseLayer(2, 3), 
                   ReLU(),
                   DenseLayer(3, 1),
                   Sigmoid()]

moons_layers =  [DenseLayer(2, 3), 
                 ReLU(),
                 DenseLayer(3, 3),
                 ReLU(),
                 DenseLayer(3, 1),
                 Sigmoid()]

xor_layers = [DenseLayer(2, 2),
              Sigmoid(),
              DenseLayer(2, 1),
              Sigmoid()]

blobs_layers = [DenseLayer(2, 3), 
                ReLU(),
                DenseLayer(3, 3),
                ReLU(),
                DenseLayer(3, 3),
                Softmax()]

datasets_dict = {
    "circles": {"layers": circles_layers, "data": make_circles(n_samples=300, factor=0.4, noise=0.1)},
    "moons": {"layers": moons_layers, "data": make_moons(n_samples=200, noise=0.1)},
    "xor": {"layers": xor_layers, "data": make_xor()},
    "blobs": {"layers": blobs_layers, "data": make_blobs(n_samples=300, n_features=2, centers=3, cluster_std=1.2)},
}

# Nonlinear circles dataset

In [ ]:
dataset = "circles"

X, y = datasets_dict[dataset]["data"]
plot_2d_clf_problem(X, y)

# Training classification model

In [ ]:
layers = datasets_dict[dataset]["layers"]
model = SequentialModel(layers=layers, loss_function=BCE_Loss(), optimizer=Optimizer_SGD(learning_rate=5e-3))

model.train(X, y.reshape(-1, 1), print_every=500, epochs=2000)

print(f'Accuracy: {np.mean(y.reshape(-1, 1) == np.round(model.predict(X)))}')

plot_2d_clf_problem(X, y, lambda x: model.predict(x) > 0.5)

# Extract each layer's output

In [ ]:
model.forward(X)

Z1 = model.wrapper.layers[0].output.copy() # first dense layer (N, 3)
A1 = model.wrapper.layers[1].output.copy() # first dense + relu (N, 3)
Z2 = model.wrapper.layers[2].output.copy() # second dense (N, 1)
A2 = model.wrapper.layers[3].output.copy() # second dense + sigmoid (N, 1)
#Z3 = model.wrapper.layers[4].output.copy() # third dense (N, 1)
#A3 = model.wrapper.layers[5].output.copy() # third dense + activation (N, 1)

# Plotting first linear transformation

In [ ]:
plot_3d_classification_output(Z1, y, "2D → 3D using first Dense Layer")

# Plotting first linear transformation and first activation

In [ ]:
plot_3d_classification_output(A1, y, "2D → 3D using first Dense Layer + ReLU")

# Plotting second linear transformation

In [ ]:
plot_1d_classification_output(Z2, y, "3D → 1D using second Dense Layer", logits=True)


# Plotting second linear transformation and second activation

In [ ]:
plot_1d_classification_output(A2, y, "3D → 1D using second Dense Layer + Sigmoid")